# 01. Ingestia paczki obrazów do warstwy Bronze
### Instalacja wymaganych bibliotek
Instalujemy pakiety niezbędne do strumieniowego pobierania zbioru danych z Hugging Face (`datasets`, `huggingface_hub`, `fsspec`), przetwarzania obrazów (`pillow`, `numpy`) oraz obsługi konfiguracji (`pyyaml`), a następnie restartujemy środowisko Pythona.

In [0]:
%pip install -U datasets huggingface_hub fsspec pillow pyyaml numpy
dbutils.library.restartPython()

### Wczytanie konfiguracji i odczyt stanu potoku (`pipeline_state`)
Wczytujemy plik `config.yaml`, upewniamy się, że Volume oraz tabela stanu `pipeline_state` istnieją w Unity Catalog, a następnie sprawdzamy numer ostatnio przetworzonej paczki (`last_batch_id`) oraz przesunięcie (`last_offset`), aby wyznaczyć zakres indeksów zdjęć do pobrania w bieżącej paczce.

In [0]:
import yaml
import os

# 1. Wczytujemy config.yaml (leży w tym samym folderze lub piętro wyżej)
config_path = "config.yaml" if os.path.exists("config.yaml") else "../config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["storage"]["catalog_name"]
SCHEMA = config["storage"]["schema_name"]
VOLUME = config["storage"]["volume_name"]
BATCH_SIZE = config["pipeline"]["batch_size"]
HF_DATASET = config["dataset"]["hf_name"]

# Ścieżka do naszego wirtualnego dysku (Volume)
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
BATCH_DIR = f"{VOLUME_PATH}/current_batch"

# 2. Upewniamy się, że Volume i tabela stanu istnieją
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_state (
    dataset_name STRING,
    last_batch_id INT,
    last_offset INT,
    updated_at TIMESTAMP
)
""")

# 3. Sprawdzamy, czy mamy już wpis dla naszego zbioru danych
state_df = spark.sql(f"""
    SELECT last_batch_id, last_offset 
    FROM {CATALOG}.{SCHEMA}.pipeline_state 
    WHERE dataset_name = '{HF_DATASET}'
    ORDER BY last_batch_id DESC LIMIT 1
""").collect()

if len(state_df) == 0:
    print("Inicjalizacja nowej tabeli stanu (start od 0)...")
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_state 
        VALUES ('{HF_DATASET}', 0, 0, current_timestamp())
    """)
    current_batch_id = 1
    current_offset = 0
else:
    current_batch_id = state_df[0]["last_batch_id"] + 1
    current_offset = state_df[0]["last_offset"]

print(f"Gotowe! Będziemy pobierać Paczkę #{current_batch_id} (zdjęcia od indeksu {current_offset} do {current_offset + BATCH_SIZE}).")

### Pobranie paczki zdjęć ze strumienia Hugging Face i wyliczenie statystyk obrazu
Otwieramy strumień treningowy zbioru danych z pominięciem dotychczas pobranych rekordów (`skip(current_offset).take(BATCH_SIZE)`). Każdy obraz zapisujemy fizycznie jako plik `.jpg` w katalogu tymczasowym w Volume (`current_batch`) oraz obliczamy jego podstawowe statystyki (wymiary, średnie wartości kanałów RGB oraz jasność) na potrzeby warstwy Bronze.

In [0]:
import numpy as np
from datasets import load_dataset
from datetime import datetime

# Tworzymy folder na pliki .jpg bieżącej paczki
os.makedirs(BATCH_DIR, exist_ok=True)

# Otwieramy strumień i wycinamy tylko nasz kawałek (od current_offset, dokładnie BATCH_SIZE zdjęć)
stream_ds = load_dataset(HF_DATASET, split=config["dataset"]["train_split"], streaming=True)
batch_stream = stream_ds.skip(current_offset).take(BATCH_SIZE)

bronze_records = []

print(f"Pobieranie {BATCH_SIZE} zdjęć do {BATCH_DIR} ...")

for idx, item in enumerate(batch_stream):
    img = item[config["dataset"]["image_col"]].convert("RGB")
    label_id = int(item[config["dataset"]["label_col"]])
    
    # Nazwa klasy (np. Forest, River) wyciągnięta z oryginalnej nazwy pliku lub ID
    raw_filename = item.get(config["dataset"]["id_col"], f"img_{current_offset + idx}.jpg")
    class_name = raw_filename.split("/")[0] if "/" in raw_filename else f"class_{label_id}"
    safe_filename = f"batch{current_batch_id}_idx{current_offset + idx}_{os.path.basename(raw_filename)}"
    
    # 1. Zapisujemy fizyczny plik .jpg w Volume
    file_path = os.path.join(BATCH_DIR, safe_filename)
    img.save(file_path, format="JPEG")
    
    # 2. Liczymy szybkie statystyki obrazu do warstwy Bronze (średnie RGB i jasność)
    img_array = np.array(img, dtype=np.float32)
    mean_r, mean_g, mean_b = img_array.mean(axis=(0, 1)).tolist()
    brightness = float(img_array.mean())
    width, height = img.size
    
    bronze_records.append({
        "batch_id": current_batch_id,
        "global_idx": current_offset + idx,
        "file_path": file_path,
        "label_id": label_id,
        "class_name": class_name,
        "width": width,
        "height": height,
        "mean_r": round(mean_r, 2),
        "mean_g": round(mean_g, 2),
        "mean_b": round(mean_b, 2),
        "brightness": round(brightness, 2),
        "ingested_at": datetime.utcnow()
    })

print(f"Pobrano i zapisano {len(bronze_records)} zdjęć!")

### Zapis metadanych i statystyk do tabeli Delta (`bronze_batch_stats`)
Zapisujemy zebrane metadane oraz statystyki kolorystyczne pobranych zdjęć w trybie `append` do tabeli Delta `bronze_batch_stats` w Unity Catalog, a następnie wyświetlamy podsumowanie liczebności i średnich wartości RGB w podziale na klasy dla bieżącej paczki.

In [0]:
# 4. Zapisujemy listę bronze_records jako tabelę Delta w Unity Catalog
bronze_df = spark.createDataFrame(bronze_records)
bronze_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_batch_stats")

print(f"Zapisano metadane w tabeli {CATALOG}.{SCHEMA}.bronze_batch_stats!")

# Podgląd pobranych klas i ich średnich kolorów RGB
display(
    spark.sql(f"""
        SELECT 
            class_name,
            label_id,
            COUNT(*) as image_count,
            ROUND(AVG(mean_r), 1) as avg_red,
            ROUND(AVG(mean_g), 1) as avg_green,
            ROUND(AVG(mean_b), 1) as avg_blue,
            ROUND(AVG(brightness), 1) as avg_brightness
        FROM {CATALOG}.{SCHEMA}.bronze_batch_stats
        WHERE batch_id = {current_batch_id}
        GROUP BY class_name, label_id
        ORDER BY label_id
    """)
)